# Synthetic control lab: what did terrorism cost the Basque Country?
**Econ 521 · Friday lab · Fri 13 Nov 2026 · Joel Reyes Mora**

Thursday's lecture motivated synthetic control with California's Proposition 99. Today we go back to the paper that introduced the method: **Abadie & Gardeazabal (2003), "The Economic Costs of Conflict: A Case Study of the Basque Country,"** *AER* 93(1).

**The question.** ETA's terrorist campaign started at the end of the 1960s and became large-scale in the mid-1970s. In the early 1960s the Basque Country was one of Spain's three richest regions. How much lower was its GDP per capita because of the conflict? There is one treated region, so we need a counterfactual built from the other 16 Spanish regions.

**By the end of the lab you should be able to:**
1. Build the paper's synthetic Basque Country step by step and reproduce its weights (Catalonia 0.85, Madrid 0.15), its Table 3 and its Figure 1.
2. Explain the **two nested problems**: the weights $W$ for a given importance matrix $V$, and the choice of $V$.
3. Read the weights as the **identity of the counterfactual**, and see why non-negative weights that sum to one matter, by comparing five ways to weight the same donor regions.
4. Run the alternative the authors describe in their Appendix B, and argue which version you would report.

| Section | Topic | Minutes |
|---|---|---|
| 0 | Setup | 3 |
| 1 | Why simple comparisons fail | 5 |
| 2 | The paper's ingredients: predictors and the convex hull (TODO 1) | 6 |
| 3 | The inner problem: weights for a given $V$ (TODO 2) | 10 |
| 4 | The outer problem: choosing $V$ as the paper does (TODO 3) | 7 |
| 5 | Why these weights? Five counterfactuals for one region | 8 |
| 6 | Another way: the paper's Appendix B (TODO 4) | 8 |
| 7 | Wrap-up and exit ticket | 3 |

## 0 · Setup
Put `basque.csv` (posted on Canvas) next to this notebook. It is the replication file that ships with the R `Synth` package: 17 regions plus Spain as a whole, 1955–1997. GDP per capita is in **thousands of 1986 US dollars**.

In [ ]:
# !pip install numpy pandas scipy matplotlib   # uncomment if needed
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
BLUE, ORANGE, GREY, GREEN, PURPLE, RED = "#2b6cb0", "#dd6b20", "#a0aec0", "#2f855a", "#6b46c1", "#c53030"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

NAMES = {"Basque Country (Pais Vasco)": "Basque Country", "Cataluna": "Catalonia", "Madrid (Comunidad De)": "Madrid",
         "Baleares (Islas)": "Balearic Islands", "Rioja (La)": "La Rioja", "Navarra (Comunidad Foral De)": "Navarre",
         "Principado De Asturias": "Asturias", "Murcia (Region de)": "Murcia", "Comunidad Valenciana": "Valencia",
         "Castilla Y Leon": "Castile and Leon", "Castilla-La Mancha": "Castile-La Mancha", "Andalucia": "Andalusia",
         "Canarias": "Canary Islands", "Cantabria": "Cantabria", "Extremadura": "Extremadura", "Galicia": "Galicia",
         "Aragon": "Aragon", "Spain (Espana)": "Spain"}

path = Path("basque.csv") if Path("basque.csv").exists() else Path("data/basque.csv")
raw = pd.read_csv(path)
raw["region"] = raw["regionname"].map(NAMES)
df = raw[raw["region"] != "Spain"].copy()           # 17 regions
Y = df.pivot(index="year", columns="region", values="gdpcap")   # years x regions

TREATED = "Basque Country"
DONORS = [r for r in Y.columns if r != TREATED]   # the donor pool: the other 16 regions
print(f"{Y.shape[1]} regions, {Y.index.min()}-{Y.index.max()}; donor pool has {len(DONORS)} regions")

In [ ]:
import hashlib as _hl, json as _json
_ANS = _json.loads('{"n_outside": {"s": 0.01, "h": "de4d6f7851a83a4e"}, "loss_equal": {"s": 0.007959850000000001, "h": "7d7496c8b9a2a433"}, "gap_paper": {"s": 0.053445, "h": "0d7513c7dfd099d2"}, "gap_outcome_only": {"s": 0.08397500000000001, "h": "1edc8a00fe8a212f"}}')

def check(name, value):
    """Prints ✓ or ✗ for a TODO answer. Only hashes are stored, so the answers are not in the notebook."""
    if value is None:
        print(f"[{name}] not answered yet")
        return
    try:
        v = float(value)
    except Exception:
        print(f"[{name}] ✗ expected a single number")
        return
    rec = _ANS[name]
    k = int(round(v / rec["s"]))
    ok = any(_hl.sha256(f"econ521-f26|{name}|{k + d}".encode()).hexdigest()[:16] == rec["h"] for d in (-1, 0, 1))
    print(f"[{name}] {'✓ correct' if ok else '✗ not quite, check your code'}")

## 1 · Why simple comparisons fail (≈ 5 min)

In [ ]:
avg16 = Y[DONORS].mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
for r in DONORS:
    ax.plot(Y.index, Y[r], color=GREY, lw=0.8)
ax.plot(Y.index, Y[TREATED], color=BLUE, lw=2.5, label="Basque Country")
ax.plot(Y.index, avg16, color=ORANGE, lw=2, ls="--", label="average of the other 16")
ax.axvline(1970, color="k", ls=":", lw=1)
ax.set_ylabel("GDP per capita (1986 USD, thousands)"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Levels: the Basque Country starts near the top", fontsize=10)
ax = axes[1]
ax.plot(Y.index, Y[TREATED] - avg16, color=BLUE, lw=2)
ax.axhline(0, color=GREY, lw=0.8); ax.axvline(1970, color="k", ls=":", lw=1)
ax.set_ylabel("Basque minus average (thousands)")
ax.set_title("Basque minus average, in thousands of USD", fontsize=10)
plt.tight_layout(); plt.show()

rank = Y.loc[1960:1969].mean().rank(ascending=False).astype(int)
print("Rank in average GDP per capita, 1960-69 (of 17):", rank[TREATED])

Two problems are visible before any terrorism:
- **Levels.** The Basque Country is about \$1,700 richer than the average region. A plain average is a poor stand-in.
- **Trends.** A design that removes only the level difference (difference-in-differences) needs the Basque Country and the comparison group to share a trend. With one rich region and 16 mostly poorer ones, that is a strong assumption. You will test it in the assignment, so we do not settle it here.

Abadie and Gardeazabal's answer: find a **weighted combination** of regions that looks like the Basque Country *before* terrorism, both in the factors that drive growth and in its GDP path.

> **Quick think.** The most similar single region is Catalonia: rich, industrial, with a strong regional identity. Why not just compare the Basque Country with Catalonia? Keep your answer; we come back to it in Section 5.

## 2 · The paper's ingredients: predictors and the convex hull (≈ 6 min)

The weights are chosen to match the Basque Country on **pre-terrorism growth predictors** (the paper's Table 3). The table lists each predictor and the years it is averaged over, exactly as in the paper:

| Predictor | Years | Unit |
|---|---|---|
| Real GDP per capita | 1960–69 | thousands of 1986 USD |
| Investment ratio | 1964–69 | % of GDP |
| Population density | 1969 | persons per km² |
| Six sectoral shares of production | 1961–69 (odd years) | % |
| Four human-capital shares of the working-age population | 1964–69 | % |

In the data the human-capital variables are counts of people. The paper reports percentages and puts "high school" and "post-high school" together in its top category. The function below follows those choices.

In [ ]:
SEC = ["sec.agriculture", "sec.energy", "sec.industry", "sec.construction", "sec.services.venta", "sec.services.nonventa"]
SCH = ["school.illit", "school.prim", "school.med", "school.high", "school.post.high"]

def make_predictors(d):
    '''Pre-terrorism predictors (one row per region), as in Abadie & Gardeazabal (2003), Table 3.'''
    out = pd.DataFrame({
        "gdpcap": d[d.year.between(1960, 1969)].groupby("region")["gdpcap"].mean(),
        "invest": d[d.year.between(1964, 1969)].groupby("region")["invest"].mean(),
        "popdens": d[d.year == 1969].set_index("region")["popdens"],
    })
    out = out.join(d[d.year.isin(range(1961, 1970, 2))].groupby("region")[SEC].mean())
    sch = d[d.year.between(1964, 1969)].groupby("region")[SCH].mean()
    sch["school.high"] = sch["school.high"] + sch.pop("school.post.high")    # 'more than high school'
    return out.join(100 * sch.div(sch.sum(axis=1), axis=0))                 # counts -> percentages

P = make_predictors(df)                       # 17 regions x 13 predictors
P_spain = make_predictors(raw[raw.region == "Spain"])

table3 = pd.DataFrame({"Basque Country": P.loc[TREATED], "Spain": P_spain.loc["Spain"]})
table3

These are the paper's Table 3 columns (1) and (2): GDP per capita 5.29 against 3.63 (the paper prints \$5,285 against \$3,633), industry 45% against 27%, and so on. The two small differences in the schooling rows come from how the data were interpolated.

**The convex hull.** A synthetic control is a weighted average with $w_j \ge 0$ and $\sum_j w_j = 1$. So for **any** predictor it can only produce values between the smallest and largest donor values. If the Basque Country lies outside that range, no synthetic control can match it on that predictor.

### TODO 1 · On how many predictors is the Basque Country outside the donors' range?

In [ ]:
X1 = P.loc[TREATED].to_numpy(float)       # 13 predictors for the Basque Country
X0 = P.loc[DONORS].to_numpy(float).T      # 13 x 16 matrix for the donors (one column per region)

outside = None    # TODO: boolean array of length 13: X1 above the donors' max OR below their min
n_outside = None  # TODO: how many True
check("n_outside", n_outside)

In [ ]:
hull = pd.DataFrame({"Basque": X1, "donor min": X0.min(axis=1), "donor max": X0.max(axis=1)}, index=P.columns)
hull["outside?"] = (hull["Basque"] > hull["donor max"]) | (hull["Basque"] < hull["donor min"])
hull.loc[["gdpcap", "sec.industry", "sec.services.nonventa", "school.illit"]]

The Basque Country had the **highest industrial share in Spain** (45% against Catalonia's 40%). The paper says so explicitly: no convex combination can reproduce it. Inside every predictor's range is necessary but not sufficient: all 13 predictors must be matched *jointly*.

## 3 · The inner problem: weights for a given $V$ (≈ 10 min)

For a given diagonal matrix $V = \text{diag}(v_1,\dots,v_K)$, the weights solve

$$W^*(V) = \arg\min_{W}\; \sum_{m=1}^{K} v_m\Big(X_{1m} - \sum_{j} w_j X_{0jm}\Big)^2 \quad\text{s.t.}\quad w_j \ge 0,\;\; \sum_j w_j = 1.$$

$v_m$ says how much predictor $m$ counts in the fit. Predictors have very different units (population density runs to the hundreds, shares to the tens), so as in the `Synth` package, each predictor is first **divided by its standard deviation across the 17 regions**.

In [ ]:
sd = P.std(ddof=1).to_numpy()
X1s, X0s = X1 / sd, X0 / sd[:, None]       # standardized predictors
K, J = X0s.shape
print(f"K = {K} predictors, J = {J} donor regions")

### TODO 2 · Write the predictor-fit loss
Write `predictor_loss(w, v)`, the objective above in standardized units. Then evaluate it at **equal weights** ($w_j = 1/16$) with **equal importance** ($v_m = 1/13$).

In [ ]:
def predictor_loss(w, v):
    '''sum_m v_m (X1s_m - sum_j w_j X0s_mj)^2'''
    return None   # TODO

loss_equal = predictor_loss(np.full(J, 1 / J), np.full(K, 1 / K))
check("loss_equal", loss_equal)

Minimizing this loss over the simplex is a small quadratic program. The function below solves it with `scipy`. Expanding the square gives $w'Hw - 2g'w$ plus a constant, which is the form the solver uses.

In [ ]:
def solve_w(v, X1s=X1s, X0s=X0s):
    '''Synthetic control weights W*(V) for a diagonal V with diagonal v.'''
    J = X0s.shape[1]
    A_, a_ = X0s * np.sqrt(v)[:, None], X1s * np.sqrt(v)
    H, g = A_.T @ A_, A_.T @ a_
    res = minimize(lambda w: w @ H @ w - 2 * g @ w, np.full(J, 1 / J), jac=lambda w: 2 * H @ w - 2 * g,
                   method="SLSQP", bounds=[(0, 1)] * J,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(J)}],
                   options={"ftol": 1e-14, "maxiter": 1000})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

def show_w(w, names=DONORS, tol=1e-3, digits=3):
    s = pd.Series(w, index=names)
    return s[s.abs() > tol].sort_values(ascending=False).round(digits)

def fmt_w(w, digits=3):
    return ", ".join(f"{k} {v:.{digits}f}" for k, v in show_w(w, digits=digits).items())

def pre_rmspe(w, years=range(1960, 1970)):
    '''Root mean squared gap in GDP per capita, Basque minus synthetic, over the given years.'''
    yrs = list(years)
    return np.sqrt(np.mean((Y.loc[yrs, TREATED] - Y.loc[yrs, DONORS] @ w) ** 2))

w_eqV = solve_w(np.full(K, 1 / K))
print("Weights when every predictor counts equally:"); print(show_w(w_eqV))
print(f"Pre-1970 RMSPE of GDP per capita: {pre_rmspe(w_eqV):.3f} (thousand USD)")

With equal importance for every predictor, the synthetic Basque Country is mostly Catalonia, plus Cantabria (an industrial neighbour) and a little Madrid. The covariates fit reasonably, but the GDP path is off by about \$500 a year in the 1960s. Change $V$ and the weights change:

In [ ]:
def v_only(names):
    v = np.array([1.0 if c in names else 1e-6 for c in P.columns])
    return v / v.sum()

for label, cols in [("GDP per capita only", ["gdpcap"]),
                    ("industry + human capital only", ["sec.industry", "school.illit", "school.prim", "school.med", "school.high"]),
                    ("investment + population density only", ["invest", "popdens"])]:
    w = solve_w(v_only(cols))
    print(f"{label:38s} -> {fmt_w(w)}   (pre-RMSPE {pre_rmspe(w):.3f})")

**The weights depend on $V$, and $V$ is a researcher choice.** One option is to set $V$ from prior knowledge of what drives growth, which the paper calls the "subjective" option. Abadie and Gardeazabal take a more data-driven route.

## 4 · The outer problem: choosing $V$ as the paper does (≈ 7 min)

The paper (Section II.A and Appendix B) picks the $V$ whose weights best reproduce the Basque **GDP per capita path during the 1960s**:

$$V^* = \arg\min_{V}\; \sum_{t=1960}^{1969}\Big(Y_{1t} - \sum_j w_j^*(V)\,Y_{jt}\Big)^2, \qquad W^* = W^*(V^*).$$

This is a **nested** optimization: every candidate $V$ calls the inner problem from Section 3. We search over $V$ with Nelder–Mead, writing $v_m = e^{\theta_m}/\sum_k e^{\theta_k}$ so that $V$ stays positive and normalized. (The scale of $V$ does not matter.) It takes a second or two.

In [ ]:
Z1 = Y.loc[1960:1969, TREATED].to_numpy()        # Basque GDP per capita, 1960-69
Z0 = Y.loc[1960:1969, DONORS].to_numpy()         # 10 x 16 donor GDP per capita

def outer_loss(theta):
    v = np.exp(theta) / np.exp(theta).sum()
    return np.mean((Z1 - Z0 @ solve_w(v)) ** 2)

res_V = minimize(outer_loss, np.zeros(K), method="Nelder-Mead",
                 options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-10})
v_star = np.exp(res_V.x) / np.exp(res_V.x).sum()
w_star = solve_w(v_star)

print("Synthetic Basque Country weights (paper: Catalonia 0.8508, Madrid 0.1492):")
print(fmt_w(w_star, digits=4))
print(f"\nPre-1970 RMSPE: {pre_rmspe(w_star):.3f}")
print("\nV* (importance of each predictor):")
print(pd.Series(v_star, index=P.columns).round(3).sort_values(ascending=False).head(5))

We reproduce the paper's weights to four decimals: **Catalonia 0.8508, Madrid 0.1492**, and zero for the other 14 regions.

Notice where $V^*$ puts its weight: about 98% on GDP per capita, with a little on industry and population density. A $V$ chosen to fit the outcome path ends up favouring the predictor that *is* the outcome. Keep this in mind for Section 6.

Now Table 3, column (3): how well does the synthetic region match the predictors?

In [ ]:
table3["Synthetic Basque"] = X0 @ w_star
table3["Average of 16"] = X0.mean(axis=1)
table3[["Basque Country", "Synthetic Basque", "Spain", "Average of 16"]]

The synthetic region matches GDP per capita almost exactly (5.27 against 5.29). It is also much closer than Spain on industry, agriculture, population density and schooling. It is **not** closer on investment, energy or market services: the weights trade those off against the predictors $V^*$ cares about. Industry is 37.6% against 45.1%, which we saw is impossible to close. These are the paper's numbers (investment 21.58, density 196.28, industry 37.64).

### TODO 3 · The estimate: Figure 1 and the average gap
Compute the synthetic GDP path for every year, $\hat Y^N_{1t} = \sum_j w_j^* Y_{jt}$. Then compute the gap **as a percentage of Basque GDP per capita**, $100\,(Y_{1t}-\hat Y^N_{1t})/Y_{1t}$, as in the paper's Figure 2, and its average over 1980–1997.

In [ ]:
synth_paper = None   # TODO: pd.Series indexed by year: Y[DONORS] @ w_star
gap_pct = None       # TODO: 100 * (Y[TREATED] - synth_paper) / Y[TREATED]
gap_paper = None     # TODO: average of gap_pct over 1980-1997
check("gap_paper", gap_paper)

In [ ]:
synth_star = Y[DONORS] @ w_star          # recomputed so the figure runs even if TODO 3 is blank
gap_star = 100 * (Y[TREATED] - synth_star) / Y[TREATED]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.plot(Y.index, Y[TREATED], color=BLUE, lw=2.5, label="Basque Country")
ax.plot(Y.index, synth_star, color=ORANGE, lw=2, ls="--", label="synthetic Basque (0.85 Catalonia + 0.15 Madrid)")
ax.axvspan(1960, 1969, color=GREY, alpha=0.15); ax.text(1960.3, Y[TREATED].max() * 0.95, "fit window", fontsize=8)
ax.set_ylabel("GDP per capita (1986 USD, thousands)"); ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("Replication of Figure 1", fontsize=10)
ax = axes[1]
ax.bar(gap_star.index, gap_star, color=np.where(gap_star < 0, BLUE, GREY))
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("gap (% of Basque GDP per capita)")
ax.set_title("Basque minus synthetic", fontsize=10)
plt.tight_layout(); plt.show()
print(gap_star.loc[[1970, 1975, 1980, 1985, 1990, 1995, 1997]].round(1).to_string())

The two lines track each other until about 1975, when ETA's campaign becomes large-scale. From the late 1970s the Basque Country falls about 10% below its synthetic counterpart, which is the paper's headline. The gap narrows a little in the mid-1990s.

**Is the answer fragile to $V$? (Optional, if time.)** The paper reports (footnote 10) that small changes to $V$ do not change the result much. You can check that claim in one cell: mix $V^*$ with random noise and re-solve.

In [ ]:
rng = np.random.default_rng(13)
w_cat, gaps = [], []
for _ in range(200):
    v_pert = 0.9 * v_star + 0.1 * rng.dirichlet(np.ones(K))     # 10% random perturbation of V*
    w = solve_w(v_pert)
    w_cat.append(w[DONORS.index("Catalonia")])
    gaps.append((100 * (Y[TREATED] - Y[DONORS] @ w) / Y[TREATED]).loc[1980:1997].mean())
print(f"Catalonia weight across 200 perturbed V's: 5th-95th percentile {np.percentile(w_cat, 5):.2f}-{np.percentile(w_cat, 95):.2f}")
print(f"Average 1980-97 gap: {np.percentile(gaps, 5):.1f}% to {np.percentile(gaps, 95):.1f}%")

## 5 · Why these weights? Five counterfactuals for one region (≈ 8 min)

Every estimator below predicts the untreated Basque GDP path as $\sum_j w_j Y_{jt}$. **They differ only in the weights.** This is the same point as last week: an estimator is a choice of weights on the comparison units.

| Counterfactual | Weights |
|---|---|
| (a) Average of 16 regions | $w_j = 1/16$ |
| (b) Nearest single region | all weight on Catalonia, as in 1-NN matching last week |
| (c) Synthetic, equal $V$ | Section 3 |
| (d) Synthetic, paper's $V^*$ | Section 4 |
| (e) Regression weights | $W^{reg} = \bar X_0'(\bar X_0\bar X_0')^{-1}\bar X_1$ from the lecture (a constant added to $X$); they sum to one but **can be negative** |

In [ ]:
X0bar = np.vstack([np.ones(J), X0])
X1bar = np.r_[1.0, X1]
w_reg = X0bar.T @ np.linalg.solve(X0bar @ X0bar.T, X1bar)

W_all = {"(a) average of 16": np.full(J, 1 / J),
         "(b) Catalonia only": np.array([1.0 if d == "Catalonia" else 0.0 for d in DONORS]),
         "(c) synthetic, equal V": w_eqV,
         "(d) synthetic, paper's V*": w_star,
         "(e) regression weights": w_reg}

rows = {}
for lab, w in W_all.items():
    g = 100 * (Y[TREATED] - Y[DONORS] @ w) / Y[TREATED]
    rows[lab] = {"sum of w": w.sum(), "# negative": int((w < -1e-9).sum()), "min w": w.min(), "max w": w.max(),
                 "max |X1 - X0w| (in SDs)": np.max(np.abs(X1s - X0s @ w)),
                 "pre-1970 RMSPE": pre_rmspe(w), "avg gap 1980-97 (%)": g.loc[1980:1997].mean()}
spectrum = pd.DataFrame(rows).T
spectrum

In [ ]:
cols = [ORANGE, GREEN, PURPLE, BLUE, RED]
fig = plt.figure(figsize=(12, 4.2))
ax = fig.add_subplot(1, 2, 1)
ax.plot(Y.index, Y[TREATED], color="k", lw=2.5, label="Basque Country (actual)")
for (lab, w), c in zip(W_all.items(), cols):
    ax.plot(Y.index, Y[DONORS] @ w, color=c, lw=1.5, ls="--", label=lab)
ax.axvline(1970, color="k", ls=":", lw=1)
ax.set_ylabel("GDP per capita (1986 USD, thousands)"); ax.legend(frameon=False, fontsize=7)
ax.set_title("Same donors, five counterfactuals", fontsize=10)
ax = fig.add_subplot(1, 2, 2)
show = ["(c) synthetic, equal V", "(d) synthetic, paper's V*", "(e) regression weights"]
x = np.arange(J); width = 0.27
for k, lab in enumerate(show):
    ax.bar(x + (k - 1) * width, W_all[lab], width, color=cols[2 + k], label=lab)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(DONORS, rotation=75, fontsize=7)
ax.set_ylabel("weight"); ax.legend(frameon=False, fontsize=7)
ax.set_title("Who builds the counterfactual?", fontsize=10)
plt.tight_layout(); plt.show()

**Read the table and the figure with your neighbour (4 min):**
1. **(a) Average.** Why is the "effect" positive and large? Which of the two problems from Section 1 does this show?
2. **(e) Regression.** These weights match all 13 predictors **exactly** (largest gap 0.00 SDs), yet they get the 1960s GDP path wrong and give an effect with the **opposite sign**. How many regions get negative weight? What does a weight of −0.6 on Castile and León mean as a counterfactual? Footnote 9 of the paper makes this point: without the non-negativity and adding-up constraints, enough donors let you fit $X_1$ perfectly, however far it is from the donors.
3. **(b) vs (d).** Catalonia alone gives almost the same average gap as the paper. So why bother with the synthetic version? (Look at the pre-1970 fit, and think about anything that happened to Catalonia, and only Catalonia, after 1970.)

## 6 · Another way: the paper's Appendix B (≈ 8 min)

Appendix B of the paper describes an alternative: skip the growth predictors and choose $W$ to reproduce **only the Basque GDP path in the 1960s**, year by year:

$$W^{out} = \arg\min_{W}\; \sum_{t=1960}^{1969}\Big(Y_{1t} - \sum_j w_j Y_{jt}\Big)^2 \quad\text{s.t.}\quad w_j\ge 0,\;\sum_j w_j = 1.$$

There is no $V$ to choose, because the "predictors" are the ten annual outcomes, each counting equally. The authors report that this version gives a larger gap. Let's check.

### TODO 4 · Build the outcome-only synthetic control
`simplex_ls(target, M)` solves $\min_w \|target - Mw\|^2$ over the simplex. Use it with `Z1` and `Z0` from Section 4.

In [ ]:
def simplex_ls(target, M):
    '''argmin_w ||target - M w||^2  subject to  w >= 0, sum(w) = 1.'''
    J = M.shape[1]
    H, g = M.T @ M, M.T @ target
    res = minimize(lambda w: w @ H @ w - 2 * g @ w, np.full(J, 1 / J), jac=lambda w: 2 * H @ w - 2 * g,
                   method="SLSQP", bounds=[(0, 1)] * J,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(J)}],
                   options={"ftol": 1e-15, "maxiter": 1000})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

In [ ]:
w_out = None             # TODO: outcome-only weights
gap_outcome_only = None  # TODO: average 1980-97 gap (% of Basque GDP per capita) using w_out
check("gap_outcome_only", gap_outcome_only)

In [ ]:
w_o = simplex_ls(Z1, Z0)     # recomputed so this cell runs even if TODO 4 is blank
comp = pd.DataFrame({"Basque Country": X1, "paper's synthetic": X0 @ w_star, "outcome-only synthetic": X0 @ w_o},
                    index=P.columns)
print("Weights  - paper:", fmt_w(w_star), "| outcome-only:", fmt_w(w_o))
print(f"Pre-1970 RMSPE - paper: {pre_rmspe(w_star):.3f} | outcome-only: {pre_rmspe(w_o):.3f}")
comp.loc[["gdpcap", "invest", "popdens", "sec.agriculture", "sec.industry", "sec.services.venta", "school.illit"]]

In [ ]:
g_o = 100 * (Y[TREATED] - Y[DONORS] @ w_o) / Y[TREATED]
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(gap_star.index, gap_star, color=BLUE, lw=2, label="paper: growth predictors + V*")
ax.plot(g_o.index, g_o, color=RED, lw=2, ls="--", label="Appendix B: 1960s GDP path only")
ax.axhline(0, color="k", lw=0.8); ax.axvspan(1960, 1969, color=GREY, alpha=0.15)
ax.set_ylabel("gap (% of Basque GDP per capita)"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Two synthetic Basque Countries, two answers", fontsize=10)
plt.tight_layout(); plt.show()

The outcome-only version **fits the 1960s better** (RMSPE 0.06 against 0.09). Yet it is built from Madrid, the **Balearic Islands** and La Rioja, and its estimated cost of conflict is about **17%** instead of 11%.

> **Discuss (3 min): which would you report?**
> - Look at the predictor table. What kind of economy is the outcome-only synthetic? (Industry share? Market services? Agriculture?) What kind of economy was the Basque Country?
> - The 1970s and early 1980s brought oil shocks and the decline of heavy industry across Europe. Which synthetic control is more exposed to that shock, the same way the Basque Country was?
> - Use the lecture's factor model, $Y^N_{jt} = \delta_t + \theta_t Z_j + \lambda_t\mu_j + \varepsilon_{jt}$. Matching the 1960s GDP path pins down the factor loadings $\mu_j$ that mattered *in the 1960s*. Why might matching the growth predictors $Z_j$ (sector shares, human capital) also matter for what happened *after* 1975?

## 7 · Wrap-up (≈ 3 min)

1. **A synthetic control is a set of weights**: non-negative, summing to one, usually sparse. Here the weights say "85% Catalonia, 15% Madrid", a counterfactual you can name and question.
2. **Two nested problems.** For a given $V$, the weights solve a constrained least-squares fit to the predictors. $V$ itself was chosen so that the weights reproduce 1960s GDP. Change $V$ and you change the counterfactual.
3. **The constraints do real work.** Unconstrained (regression) weights match every predictor perfectly and still give a very different answer, through extrapolation and negative weights.
4. **Fit is not everything.** The best pre-period fit (outcome-only) builds the Basque Country from tourism regions and gives a larger estimate. Which characteristics you match decides which post-1975 shocks your counterfactual shares.
5. **Not covered today:** how sure are we? The paper ran a placebo study on Catalonia, and the lecture showed permutation inference. You will design and test your *own* synthetic control in the assignment.

### Exit ticket (on Canvas)
1. In one sentence, what does "Catalonia 0.85, Madrid 0.15" mean for the counterfactual?
2. Why can regression weights match all 13 predictors exactly while synthetic control cannot? Why is the exact match not reassuring?
3. The outcome-only synthetic control fits the 1960s better but gives a 17% cost instead of 11%. Give one reason to prefer the paper's version.